# Purpose:
- To filter activated landmarks 
    - When there are too many, it slows down BigWarp too much
- Do not filter the ones near the edge of the volume
    - These are critical landmarks.
# Protocol:
- Read current landmark.
- Find near-edge landmarks
    - In czstack centroids, 50 um from the edges, 100 um from the bottom of the volume
- Divide the rest of the volume to desired subvolumes
- Greedy approach within the volume
- Parallelize the process (first, test single processing)
- Save matched points dataFrame.
- Uncheck filtered pairs

In [1]:
import os
import zarr
import numpy as np
import tifffile as tiff
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.interpolate import Rbf
from scipy.spatial import distance_matrix
import cv2
import os
import json
from pathlib import Path

DATA_DIR = Path('/root/capsule/data/')

%load_ext autoreload
%autoreload 2

In [2]:
def get_ids_from_landmarks(landmarks):
    columns = ['ids', 'active', 'czstack_x', 'czstack_y', 'czstack_z', 'hcr_x', 'hcr_y', 'hcr_z']
    assert len(landmarks.columns) == len(columns)
    if not all([a==b for a,b in zip(landmarks.columns, columns)]):
        landmarks.columns = columns
    matched_ids = landmarks['ids'].values
    def _get_ids(x):
        if x.startswith('cz'):
            cz_id = int(x.split('-')[0].split('cz')[-1])
            hcr_id = int(x.split('-')[1].split('hcr')[-1])
            return cz_id, hcr_id
        else:
            return -1
    
    ids = [_get_ids(x) for x in matched_ids]
    ids = [id for id in ids if id != -1]
    zstack_ids = [id[0] for id in ids]
    hcr_ids = [id[1] for id in ids]
    return zstack_ids, hcr_ids

In [16]:
# Setting
subject_id = 788406
czstack_date = '2025-07-18'
save_dir = Path(f'/root/capsule/scratch/{subject_id}_{czstack_date}_coreg_cpsam/')

###
# IMPORTANT
###
czstack_xy_size = 400 # either 400 or 700 (um in one dimension)
czstack_xy_pix = 512
czstack_z_size = 450
czstack_z_pix = 450
boundary_margins = (50, 50, 100) # (x, y, z) in um
boundary_margin_proportions = (0.1, 0.1, 0.2) # (x, y, z) in proportion to the size of the stack
current_landmark_fn = '/root/capsule/scratch/788406_2025-07-18_coreg_cpsam/788406_2025-07-18_landmarks_matched_ext_iter3.csv'

In [11]:
landmarks = pd.read_csv(current_landmark_fn, header=None)
columns = ['ids', 'active', 'czstack_x', 'czstack_y', 'czstack_z', 'hcr_x', 'hcr_y', 'hcr_z']
landmarks.columns = columns
num_roi_points = sum(landmarks.ids.str.startswith('cz'))
landmarks = landmarks.query('active')

points_zstack = landmarks[['czstack_x', 'czstack_y', 'czstack_z']].values.astype(np.float32)
points_HCR = landmarks[['hcr_x', 'hcr_y', 'hcr_z']].values.astype(np.float32)

matched_czstack_ids, matched_hcr_ids = get_ids_from_landmarks(landmarks)
print(len(matched_czstack_ids))

778


In [12]:
landmarks

,ids,active,czstack_x,czstack_y,czstack_z,hcr_x,hcr_y,hcr_z
154,cz69-hcr7483,True,330.290785,388.301024,74.401365,1054.743250,965.693632,226.000000
155,cz215-hcr14917,True,85.392700,43.191737,127.279583,1401.047318,1457.445410,329.000000
156,cz217-hcr18295,True,129.081983,388.629091,126.741818,1324.860423,957.778110,359.000000
157,cz116-hcr10230,True,332.254650,393.683451,93.897691,1054.743250,959.756991,279.000000
158,cz317-hcr27666,True,229.916900,330.267507,155.945845,1203.159279,1059.690451,437.000000
...,...,...,...,...,...,...,...,...
959,Pt-65,True,80.348714,213.632757,127.117473,1392.257200,1211.482174,349.808469
960,Pt-66,True,280.434153,265.091838,159.849694,1142.006535,1160.210600,439.330850
961,Pt-67,True,21.752589,488.314940,43.401802,1454.369242,808.388234,127.270191
962,Pt-68,True,190.436321,445.101089,400.802667,1340.762098,1008.598321,1134.272021


In [8]:
# get number of pairs activated
pair_df = landmarks[landmarks[0].str.startswith('cz')]
len(pair_df[pair_df[1]])

778

In [17]:
def filter_near_edge_landmarks(landmarks_df, 
                               czstack_xy_size, 
                               czstack_z_size,
                               boundary_margin_proportions,
                               keep_edge=True):
    """
    Filter landmarks based on proximity to volume edges.
    
    Parameters:
    -----------
    landmarks_df : pd.DataFrame
        DataFrame with columns 'czstack_x', 'czstack_y', 'czstack_z'
    czstack_xy_size : float
        Size of the volume in x and y dimensions (um)
    czstack_z_size : float
        Size of the volume in z dimension (um)
    boundary_margin_proportions : tuple
        (x_prop, y_prop, z_prop) - proportion of volume size to use as margin
    keep_edge : bool
        If True, keep only near-edge landmarks
        If False, keep only landmarks away from edges
    
    Returns:
    --------
    pd.DataFrame : filtered landmarks
    """
    # Calculate margins in um
    margin_x = boundary_margin_proportions[0] * czstack_xy_size
    margin_y = boundary_margin_proportions[1] * czstack_xy_size
    margin_z = boundary_margin_proportions[2] * czstack_z_size
    
    # Identify landmarks near edges
    near_edge_x = (landmarks_df['czstack_x'] < margin_x) | \
                  (landmarks_df['czstack_x'] > (czstack_xy_size - margin_x))
    near_edge_y = (landmarks_df['czstack_y'] < margin_y) | \
                  (landmarks_df['czstack_y'] > (czstack_xy_size - margin_y))
    near_edge_z = (landmarks_df['czstack_z'] < margin_z) | \
                  (landmarks_df['czstack_z'] > (czstack_z_size - margin_z))
    
    # Landmarks are near edge if they're near edge in ANY dimension
    near_edge = near_edge_x | near_edge_y | near_edge_z
    
    if keep_edge:
        return landmarks_df[near_edge]
    else:
        return landmarks_df[~near_edge]

# Filter to keep only near-edge landmarks
edge_landmarks = filter_near_edge_landmarks(
    landmarks,
    czstack_xy_size,
    czstack_z_size,
    boundary_margin_proportions,
    keep_edge=True
)

# Filter to keep only landmarks away from edges
interior_landmarks = filter_near_edge_landmarks(
    landmarks,
    czstack_xy_size,
    czstack_z_size,
    boundary_margin_proportions,
    keep_edge=False
)

print(f"Total landmarks: {len(landmarks)}")
print(f"Near-edge landmarks: {len(edge_landmarks)}")
print(f"Interior landmarks: {len(interior_landmarks)}")


Total landmarks: 810
Near-edge landmarks: 547
Interior landmarks: 263
